# Flatten gridded data for trend fitting

In [44]:
# import sys; sys.path.append('/rds/general/user/cb2714/home/wwa'); from wwa import *
import xarray as xr, regionmask, numpy as np, matplotlib.pyplot as plt, cartopy
xn,xx,yn,yx = [-11,30,33,66]

mapproj = cartopy.crs.AlbersEqualArea(central_longitude = 0, central_latitude = 50.0)      # need to fix missing coastlines
dataproj = cartopy.crs.PlateCarree()

# make a new directory to store the data & results
! mkdir -p flattened; mkdir -p flattened-res

## Event definitions

In [2]:
# load daily data
fpath = "/rds/general/user/cb2714/home/00_WWA_project_folder/ephemeral/euro_heat/"
ds = xr.open_dataset(fpath+"data/tmax_era5.nc").tmax.sel(lon = slice(xn,xx), lat = slice(yx,yn))

# use your event definition to get an annual time series at each grid cell
da = ds.rolling(time = 3).mean().resample(time = "YS").max()
da = da.assign_coords(time = da.time.dt.year).rename(time = "year")
da.to_netcdf("data/tx3x_era5_gridded.nc")

# use your event definition to get an annual time series at each grid cell
da = ds.rolling(time = 3).mean().groupby("time.month")[6].resample(time = "YS").max()
da = da.assign_coords(time = da.time.dt.year).rename(time = "year")
da.to_netcdf("data/tx3x-june_era5_gridded.nc")

In [3]:
# load daily data
fpath = "/rds/general/user/cb2714/home/00_WWA_project_folder/ephemeral/euro_heat/"
ds = xr.open_dataset(fpath+"data/tmin_era5.nc").tmin.sel(lon = slice(xn,xx), lat = slice(yx,yn))

# use your event definition to get an annual time series at each grid cell
da = ds.rolling(time = 3).mean().resample(time = "YS").max()
da = da.assign_coords(time = da.time.dt.year).rename(time = "year")
da.to_netcdf("data/tn3x_era5_gridded.nc")

# use your event definition to get an annual time series at each grid cell
da = ds.rolling(time = 3).mean().groupby("time.month")[6].resample(time = "YS").max()
da = da.assign_coords(time = da.time.dt.year).rename(time = "year")
da.to_netcdf("data/tn3x-june_era5_gridded.nc")

## Flatten the data and save ready for processing

In [4]:
for varnm in ["tx3x", "tx3x-june", "tn3x", "tn3x-june"]:
    fnm_root = varnm+"_era5"
    
    # relabel dates as years
    da = xr.open_dataset("data/"+fnm_root+"_gridded.nc")
    da = da[list(da.data_vars)[0]]
    
    # save the map for easier reconstruction later
    da.mean("year").to_netcdf("data/map-tmplt_"+fnm_root+".nc")
    
    # flatten & convert to data.frame
    df = da.stack(xy = ["lat", "lon"]).dropna(dim = "xy", how = "all").to_pandas()
    
    # save data.frame as .csv (split into chunks if really large)
    ncols = 2500
    if df.shape[1] > ncols:
        for i in range(int(np.ceil(df.shape[1] / ncols))):
            df.iloc[:,slice(i*ncols,(i+1)*ncols)].to_csv("flattened/"+fnm_root+"-flattened_"+str(i+1).rjust(2,"0")+".csv")
    else:
        df.to_csv("flattened/"+fnm_root+"-flattened.csv")